# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/rudrabansal10/FlyRank_Internship/blob/main/work/notebooks/w05_model.ipynb)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [61]:
from pathlib import Path

import numpy as np
import pandas as pd

from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import precision_score
from sklearn.model_selection import GroupShuffleSplit
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.tree import DecisionTreeClassifier

RANDOM_STATE = 42
TOP_K = 50

df = pd.read_csv("https://raw.githubusercontent.com/rudrabansal10/FlyRank_Internship/43b468d73eba109085f02d01f3a59754d5356453/data/raw/content_refresh_anonymized.csv")

df["is_declining_label"] = (df["trend_direction"] == "down").astype(int)

print("Rows:", len(df))
print("Clients:", df["client_id"].nunique())
print("Proxy-label rate:", round(df["is_declining_label"].mean(), 3))

Rows: 30000
Clients: 32
Proxy-label rate: 0.542


## 1. Method choice and why

*Which method from the toolkit, and why it fits your lane.*

I will use supervised binary classification to rank content pages for editorial review. The target is the starter proxy label, `is_declining_label`, which identifies pages whose observed trend direction is “down.”

This fits the Content Opportunity Scoring lane because the practical question is not simply whether a page is declining, but which pages should be reviewed first when editorial capacity is limited. Each classifier produces a probability-like score that can be used to rank pages, and performance will be evaluated using Precision@50: the share of labelled-declining pages among the 50 highest-priority pages.

I will compare three safe, interpretable model types:

- Logistic regression as the readable learned baseline.
- A shallow decision tree to capture a small number of understandable non-linear rules.
- A constrained random forest to test whether modest feature interactions improve the ranking.

All models will use the same leakage-safe feature set and the same client-grouped holdout split. The selected model will be the one with the best held-out Precision@50, compared with the frozen rule baseline.

## 2. Split design

*Grouped by client? Time-aware? Say why this split is honest for your question.*

I will use a grouped train/test split by `client_id` (80% train, 20% test, fixed random seed). No client appears in both sets, so the model is tested on entirely unseen client groups rather than learning client-specific patterns.

A time-aware split is not possible with this starter dataset because it contains only trailing-90-day aggregates, not earlier feature windows with a later outcome period. This grouped split is therefore the most honest available design, while a future warehouse-based version should use strictly earlier features and later outcomes.

## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*

In [62]:
# CREATING FEATURE FRAME

numeric_features = [
    "search_volume",
    "competition",
    "cpc",
    "word_count",
    "char_count",
    "impressions_90d",
    "clicks_90d",
    "pageviews_90d",
    "sessions_90d",
    "users_90d",
    "engaged_sessions_90d",
    "ai_sessions_90d",
    "scroll_events_90d",
    "days_with_impressions",
    "days_with_sessions",
    "content_age_days",
    "days_since_last_update",
    "ctr",
    "avg_position",
    "engagement_rate",
    "scroll_rate",
    "ai_traffic_pct",
]

categorical_features = [
    "competition_level",
    "content_type",
    "main_intent",
    "age_tier",
    "freshness_tier",
]

excluded_features = [
    "content_id",
    "client_id",
    "trend_direction",
    "trend_pct",
    "impressions_last_30d",
    "clicks_last_30d",
    "sessions_last_30d",
    "impressions_prev_30d",
    "clicks_prev_30d",
    "sessions_prev_30d",
    "provider_used",
    "model_used",
]

X = df[numeric_features + categorical_features].copy()
y = df["is_declining_label"].copy()
groups = df["client_id"].copy()

# Position 0 means no position data, not the best possible rank.
X["has_position_data"] = (X["avg_position"] > 0).astype(int)
X["avg_position"] = X["avg_position"].replace(0, np.nan)

# These transforms use only each row's own value; no held-out distribution is learned here.
log_source_columns = [
    "impressions_90d",
    "clicks_90d",
    "sessions_90d",
    "ai_sessions_90d",
]

for column in log_source_columns:
    X[f"log_{column}"] = np.log1p(X[column].clip(lower=0))

numeric_features = numeric_features + [
    "has_position_data",
    *[f"log_{column}" for column in log_source_columns],
]

assert not set(excluded_features).intersection(X.columns)
assert X.shape[0] == y.shape[0]

print("Safe feature-frame shape:", X.shape)

Safe feature-frame shape: (30000, 32)


In [63]:
# RECREATING THE EXACT SAME GROUPED HOLDOUT

splitter = GroupShuffleSplit(
    n_splits=1,
    test_size=0.20,
    random_state=RANDOM_STATE,
)

train_index, test_index = next(
    splitter.split(X, y=y, groups=groups)
)

X_train = X.iloc[train_index].copy()
X_test = X.iloc[test_index].copy()
y_train = y.iloc[train_index].copy()
y_test = y.iloc[test_index].copy()

print("Train rows:", len(X_train))
print("Held-out rows:", len(X_test))
print("Train clients:", groups.iloc[train_index].nunique())
print("Held-out clients:", groups.iloc[test_index].nunique())
print("Held-out base rate:", round(y_test.mean(), 3))

assert set(groups.iloc[train_index]).isdisjoint(set(groups.iloc[test_index]))

Train rows: 23837
Held-out rows: 6163
Train clients: 25
Held-out clients: 7
Held-out base rate: 0.511


In [64]:
# PREPROCESSING

numeric_pipeline = Pipeline(steps=[
        ("imputer", SimpleImputer(strategy="median", add_indicator=True)),
        ("scaler", StandardScaler()),
      ])

categorical_pipeline = Pipeline(steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore")),
    ])

preprocessor = ColumnTransformer(
    transformers=[
        ("numeric", numeric_pipeline, numeric_features),
        ("categorical", categorical_pipeline, categorical_features),
    ],
    remainder="drop",)

In [65]:
# Reproducing the frozen ML-07 baseline on this same test set

def precision_at_k(labels, scores, k=50):
    order = np.argsort(-np.asarray(scores))
    return float(np.asarray(labels)[order[:k]].mean())

test_baseline = df.iloc[test_index].copy()

test_baseline["low_ctr_visible"] = (
    (test_baseline["impressions_90d"] >= 500)
    & (test_baseline["avg_position"] > 0)
    & (test_baseline["avg_position"] <= 20)
    & (test_baseline["ctr"] < 0.5)
).astype(int)

# This reproduces the frozen ML-07 score calculation.
full_visibility_score = df["impressions_90d"].rank(pct=True)
test_baseline["visibility_score"] = full_visibility_score.iloc[test_index].to_numpy()

test_baseline["baseline_score"] = (
    0.70 * test_baseline["low_ctr_visible"]
    + 0.30 * test_baseline["visibility_score"]
)

baseline_p50 = precision_at_k(
    y_test,
    test_baseline["baseline_score"],
    k=TOP_K,
)

print("Held-out base rate:", round(y_test.mean(), 3))
print("Frozen baseline Precision@50:", round(baseline_p50, 3))

Held-out base rate: 0.511
Frozen baseline Precision@50: 0.42


In [72]:
# TRAINING THE 3 MODELS

models = {
    "Logistic regression": Pipeline(
        steps=[
            ("preprocessor", clone(preprocessor)),
            ("model", LogisticRegression(
                max_iter=2000,
                class_weight=None,
                random_state=RANDOM_STATE,
            )),
        ]
    ),
    "Decision tree (max_depth=3)": Pipeline(
        steps=[
            ("preprocessor", clone(preprocessor)),
            ("model", DecisionTreeClassifier(
                max_depth=3,
                min_samples_leaf=30,
                random_state=RANDOM_STATE,
            )),
        ]
    ),
    "Random forest": Pipeline(
        steps=[
            ("preprocessor", clone(preprocessor)),
            ("model", RandomForestClassifier(
                n_estimators=300,
                max_depth=3,
                min_samples_leaf=25,
                random_state=RANDOM_STATE,
                n_jobs=-1,
            )),
        ]
    ),
}

fitted_models = {}
model_results = []

for model_name, pipeline in models.items():
    pipeline.fit(X_train, y_train)
    test_scores = pipeline.predict_proba(X_test)[:, 1]
    p_at_50 = precision_at_k(y_test, test_scores, k=TOP_K)

    fitted_models[model_name] = pipeline
    model_results.append(
        {
            "method": model_name,
            "held_out_base_rate": y_test.mean(),
            "precision_at_50": p_at_50,
        }
    )

In [73]:
# COMPARISION TABLE

comparison = pd.DataFrame(
    [
        {
            "method": "Frozen rule baseline",
            "held_out_base_rate": y_test.mean(),
            "precision_at_50": baseline_p50,
        },
        *model_results,
    ]
)

comparison["held_out_base_rate"] = comparison["held_out_base_rate"].round(3)
comparison["precision_at_50"] = comparison["precision_at_50"].round(3)

comparison = comparison.sort_values(
    "precision_at_50",
    ascending=False,
).reset_index(drop=True)

best_model = comparison.loc[0, "method"]

display(comparison)
print("Best Model :", best_model,"\n",
      "(precision@50) : ",comparison.loc[0,"precision_at_50"])

,method,held_out_base_rate,precision_at_50
0,Logistic regression,0.511,0.84
1,Random forest,0.511,0.72
2,Decision tree (max_depth=3),0.511,0.56
3,Frozen rule baseline,0.511,0.42


Best Model : Logistic regression 
 (precision@50) :  0.84


In [68]:
#TOP FEATURES OF BEST MODEL

winner_name = best_model
winner = fitted_models[winner_name]

feature_names = winner.named_steps["preprocessor"].get_feature_names_out()
model = winner.named_steps["model"]

if hasattr(model, "feature_importances"):
    importance_values = model.feature_importances
else:
    importance_values = np.abs(model.coef_.ravel())

feature_importance = (
    pd.DataFrame({
            "feature": feature_names,
            "importance": importance_values,
        }
    ).sort_values("importance", ascending=False).head(10)
)

display(feature_importance)

,feature,importance
23,numeric__log_impressions_90d,1.005711
9,numeric__users_90d,0.917153
8,numeric__sessions_90d,0.740956
24,numeric__log_clicks_90d,0.589951
37,categorical__content_type_comparison article,0.484528
49,categorical__freshness_tier_181+,0.467598
22,numeric__has_position_data,0.437307
32,numeric__missingindicator_avg_position,0.437307
42,categorical__main_intent_navigational,0.359011
18,numeric__avg_position,0.296523


## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*

In [69]:
winner_scores = winner.predict_proba(X_test)[:, 1]

error_review = df.iloc[test_index][[
    "content_type",
    "main_intent",
    "impressions_90d",
    "ctr",
    "avg_position",
    "content_age_days",
    "days_since_last_update",
]].copy()

error_review["actual_proxy_label"] = y_test.to_numpy()
error_review["predicted_decline_score"] = winner_scores

false_positives = (
    error_review.loc[error_review["actual_proxy_label"] == 0]
    .sort_values("predicted_decline_score", ascending=False)
    .head()
)

display(false_positives)

,content_type,main_intent,impressions_90d,ctr,avg_position,content_age_days,days_since_last_update,actual_proxy_label,predicted_decline_score
27993,keyword article,informational,1266,0.0,4.6,106,106,0,0.932770
12869,keyword article,informational,15101,0.0,5.7,421,7,0,0.930318
26614,keyword article,informational,290,0.0,5.9,96,20,0,0.918187
25560,keyword article,informational,1463,0.0,1.5,106,8,0,0.916739
20736,keyword article,informational,3115,0.0,12.8,275,104,0,0.909004


These are high-scoring false positives: pages the model prioritized but which were not labelled declining by the starter proxy. They are difficult because the available 90-day aggregate features can resemble the observed profile of declining pages without containing a future outcome. The model should therefore support human review rather than automate refresh decisions.

# FINAL CONCLUSION

## Result

Using the same grouped client-level holdout as ML-07, the frozen rule baseline achieved Precision@50 of **0.420** against a held-out proxy-label base rate of **0.511**.

The selected model was **LOGISTIC REGRESSION** with held-out Precision@50 of ****. This is an observed ranking result on the starter proxy label, not evidence that the model predicts future search declines or that refreshing content causes recovery.

The proposed use is a ranked queue for human editorial review. A later capstone version should use earlier feature windows and a strictly later outcome window from the daily warehouse data.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.